In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


In [4]:
FILE_PATH = Path("D:\capstone1\data\sms-call-internet-mi-2013-11-01.csv")

# Load raw file
df_raw = pd.read_csv(FILE_PATH)

print("File:", FILE_PATH)
print("Shape:", df_raw.shape)

<>:1: SyntaxWarning: invalid escape sequence '\c'
<>:1: SyntaxWarning: invalid escape sequence '\c'
C:\Users\harikumar.g\AppData\Local\Temp\ipykernel_9940\2590312496.py:1: SyntaxWarning: invalid escape sequence '\c'
  FILE_PATH = Path("D:\capstone1\data\sms-call-internet-mi-2013-11-01.csv")


File: D:\capstone1\data\sms-call-internet-mi-2013-11-01.csv
Shape: (1891928, 8)


In [5]:

print("\n===== SHAPE =====")
print(df_raw.shape)

print("\n===== RAW COLUMNS =====")
print(df_raw.columns.tolist())

print("\n===== DATA TYPES =====")
print(df_raw.dtypes)

print("\n===== FIRST 5 ROWS =====")
print(df_raw.head())

print("\n===== RAW INFO =====")
df_raw.info()


===== SHAPE =====
(1891928, 8)

===== RAW COLUMNS =====
['datetime', 'CellID', 'countrycode', 'smsin', 'smsout', 'callin', 'callout', 'internet']

===== DATA TYPES =====
datetime           str
CellID           int64
countrycode      int64
smsin          float64
smsout         float64
callin         float64
callout        float64
internet       float64
dtype: object

===== FIRST 5 ROWS =====
              datetime  CellID  countrycode   smsin  smsout  callin  callout  \
0  2013-11-01 00:00:00       1            0  0.3521     NaN     NaN   0.0273   
1  2013-11-01 00:00:00       1           33     NaN     NaN     NaN      NaN   
2  2013-11-01 00:00:00       1           39  1.7322  1.1047  0.5919   0.4020   
3  2013-11-01 00:00:00       2            0  0.3581     NaN     NaN   0.0273   
4  2013-11-01 00:00:00       2           33     NaN     NaN     NaN      NaN   

   internet  
0       NaN  
1    0.0261  
2   57.7729  
3       NaN  
4    0.0274  

===== RAW INFO =====
<class 'pandas.Dat

2. Raw → canonical mapping

In [6]:
RAW_TO_CANONICAL = {
    "datetime": "timestamp",
    "CellID": "grid_id",
    "countrycode": "country_code",
    "smsin": "sms_in",
    "smsout": "sms_out",
    "callin": "call_in",
    "callout": "call_out",
    "internet": "internet_activity"
}

print("\n===== RAW → CANONICAL MAPPING =====")

for raw_col, canonical_col in RAW_TO_CANONICAL.items():
    print(f"{raw_col:15} -> {canonical_col}")


===== RAW → CANONICAL MAPPING =====
datetime        -> timestamp
CellID          -> grid_id
countrycode     -> country_code
smsin           -> sms_in
smsout          -> sms_out
callin          -> call_in
callout         -> call_out
internet        -> internet_activity


In [7]:
missing_columns = [
    col for col in RAW_TO_CANONICAL
    if col not in df_raw.columns
]

if missing_columns:
    raise ValueError(
        f"Missing expected raw columns: {missing_columns}"
    )

print("\nAll expected raw columns are present.")


All expected raw columns are present.


3. Create canonical DataFrame

In [8]:
df = df_raw.rename(
    columns=RAW_TO_CANONICAL
).copy()

print("\n===== CANONICAL COLUMNS =====")
print(df.columns.tolist())


===== CANONICAL COLUMNS =====
['timestamp', 'grid_id', 'country_code', 'sms_in', 'sms_out', 'call_in', 'call_out', 'internet_activity']


In [9]:
df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce"
)

print("\n===== TIMESTAMP DTYPE =====")
print(df["timestamp"].dtype)


===== TIMESTAMP DTYPE =====
datetime64[us]


4. Standardize data types

In [10]:
df["grid_id"] = pd.to_numeric(
    df["grid_id"],
    errors="coerce"
).astype("Int64")

df["country_code"] = pd.to_numeric(
    df["country_code"],
    errors="coerce"
).astype("Int64")

In [11]:
activity_cols = [
    "sms_in",
    "sms_out",
    "call_in",
    "call_out",
    "internet_activity"
]

for col in activity_cols:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

In [12]:
print("\n===== CANONICAL DTYPES =====")
print(df.dtypes)


===== CANONICAL DTYPES =====
timestamp            datetime64[us]
grid_id                       Int64
country_code                  Int64
sms_in                      float64
sms_out                     float64
call_in                     float64
call_out                    float64
internet_activity           float64
dtype: object


5. Verify exactly 24 timestamps

In [13]:
unique_timestamps = df["timestamp"].dropna().sort_values().unique()

print("\n===== TIMESTAMP VALIDATION =====")
print("Number of distinct timestamps:", len(unique_timestamps))

print("\nTimestamps:")
for ts in unique_timestamps:
    print(ts)


===== TIMESTAMP VALIDATION =====
Number of distinct timestamps: 24

Timestamps:
2013-11-01 00:00:00
2013-11-01 01:00:00
2013-11-01 02:00:00
2013-11-01 03:00:00
2013-11-01 04:00:00
2013-11-01 05:00:00
2013-11-01 06:00:00
2013-11-01 07:00:00
2013-11-01 08:00:00
2013-11-01 09:00:00
2013-11-01 10:00:00
2013-11-01 11:00:00
2013-11-01 12:00:00
2013-11-01 13:00:00
2013-11-01 14:00:00
2013-11-01 15:00:00
2013-11-01 16:00:00
2013-11-01 17:00:00
2013-11-01 18:00:00
2013-11-01 19:00:00
2013-11-01 20:00:00
2013-11-01 21:00:00
2013-11-01 22:00:00
2013-11-01 23:00:00


In [14]:
if len(unique_timestamps) == 24:
    print("\nPASS: Exactly 24 distinct timestamps found.")
else:
    print(
        f"\nFAIL: Expected 24 timestamps, "
        f"found {len(unique_timestamps)}."
    )


PASS: Exactly 24 distinct timestamps found.


6. Verify one-hour cadence

In [15]:
timestamps = pd.Series(unique_timestamps)

intervals = timestamps.diff().dropna()

print("\n===== TIMESTAMP INTERVALS =====")
print(intervals.value_counts())


===== TIMESTAMP INTERVALS =====
0 days 01:00:00    23
Name: count, dtype: int64


In [ ]:
expected_interval = pd.Timedelta(hours=1)

cadence_valid = (intervals == expected_interval).all()

if cadence_valid:
    print("\nPASS: All consecutive timestamps are exactly 1 hour apart.")
else:
    print("\nFAIL: Timestamp cadence is not hourly.")

    print("\nInvalid intervals:")
    print(intervals[intervals != expected_interval])


PASS: All consecutive timestamps are exactly 1 hour apart.


In [17]:
hourly_validation = {
    "distinct_timestamps": len(unique_timestamps),
    "expected_timestamps": 24,
    "exactly_24": len(unique_timestamps) == 24,
    "hourly_cadence": cadence_valid
}

print("\n===== HOURLY VALIDATION SUMMARY =====")
print(hourly_validation)


===== HOURLY VALIDATION SUMMARY =====
{'distinct_timestamps': 24, 'expected_timestamps': 24, 'exactly_24': True, 'hourly_cadence': np.True_}


7. Derive date, hour and day_of_week

In [18]:

df["date"] = df["timestamp"].dt.date
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.day_name()

print("\n===== TEMPORAL COLUMNS =====")
print(
    df[
        [
            "timestamp",
            "date",
            "hour",
            "day_of_week"
        ]
    ].head(10)
)


===== TEMPORAL COLUMNS =====
   timestamp        date  hour day_of_week
0 2013-11-01  2013-11-01     0      Friday
1 2013-11-01  2013-11-01     0      Friday
2 2013-11-01  2013-11-01     0      Friday
3 2013-11-01  2013-11-01     0      Friday
4 2013-11-01  2013-11-01     0      Friday
5 2013-11-01  2013-11-01     0      Friday
6 2013-11-01  2013-11-01     0      Friday
7 2013-11-01  2013-11-01     0      Friday
8 2013-11-01  2013-11-01     0      Friday
9 2013-11-01  2013-11-01     0      Friday


8. Missing grid_id

In [19]:
missing_grid_id = df["grid_id"].isna().sum()

print("\n===== MISSING GRID ID =====")
print("Missing grid_id:", missing_grid_id)
print(
    "Missing percentage:",
    round(df["grid_id"].isna().mean() * 100, 2),
    "%"
)


===== MISSING GRID ID =====
Missing grid_id: 0
Missing percentage: 0.0 %


9. Missing timestamp

In [20]:
missing_timestamp = df["timestamp"].isna().sum()

print("\n===== MISSING TIMESTAMP =====")
print("Missing timestamp:", missing_timestamp)
print(
    "Missing percentage:",
    round(df["timestamp"].isna().mean() * 100, 2),
    "%"
)


===== MISSING TIMESTAMP =====
Missing timestamp: 0
Missing percentage: 0.0 %


10. Blank activity measures

In [21]:
print("\n===== ACTIVITY NULL COUNTS =====")

for col in activity_cols:
    count = df[col].isna().sum()

    print(
        f"{col:20} : {count}"
    )


===== ACTIVITY NULL COUNTS =====
sms_in               : 1086153
sms_out              : 1422446
call_in              : 1407781
call_out             : 1037413
internet_activity    : 1087074


Exact duplicate rows

In [23]:
duplicate_count = df_raw.duplicated().sum()

print("\n===== EXACT DUPLICATES =====")
print("Exact duplicate rows:", duplicate_count)


===== EXACT DUPLICATES =====
Exact duplicate rows: 0


In [24]:
duplicates = df_raw[
    df_raw.duplicated(keep=False)
]

print("\nDuplicate rows:")
print(duplicates.head(20))


Duplicate rows:
Empty DataFrame
Columns: [datetime, CellID, countrycode, smsin, smsout, callin, callout, internet]
Index: []


12. Negative activity values

In [25]:

print("\n===== NEGATIVE ACTIVITY VALUES =====")

negative_counts = {}

for col in activity_cols:
    count = (df[col] < 0).sum()
    negative_counts[col] = count

    print(f"{col:20} : {count}")


===== NEGATIVE ACTIVITY VALUES =====
sms_in               : 0
sms_out              : 0
call_in              : 0
call_out             : 0
internet_activity    : 0


In [26]:
negative_rows = df[
    (df[activity_cols] < 0).any(axis=1)
]

print("\nRows containing negative activity:")
print(negative_rows.head(20))


Rows containing negative activity:
Empty DataFrame
Columns: [timestamp, grid_id, country_code, sms_in, sms_out, call_in, call_out, internet_activity, date, hour, day_of_week]
Index: []


13. Country-code rows for same grid + hour

In [28]:
grid_hour_counts = (
    df.groupby(
        ["timestamp", "grid_id"]
    )["country_code"]
    .nunique()
    .reset_index(name="country_code_count")
)

print("\n===== COUNTRY CODE COUNTS PER GRID + HOUR =====")
print(grid_hour_counts.head(20))

print(grid_hour_counts.count())


===== COUNTRY CODE COUNTS PER GRID + HOUR =====
    timestamp  grid_id  country_code_count
0  2013-11-01        1                   3
1  2013-11-01        2                   3
2  2013-11-01        3                   3
3  2013-11-01        4                   3
4  2013-11-01        5                   3
5  2013-11-01        6                   3
6  2013-11-01        7                   3
7  2013-11-01        8                   3
8  2013-11-01        9                   3
9  2013-11-01       10                   3
10 2013-11-01       11                   4
11 2013-11-01       12                   4
12 2013-11-01       13                   5
13 2013-11-01       14                   4
14 2013-11-01       15                   3
15 2013-11-01       16                   4
16 2013-11-01       17                   3
17 2013-11-01       18                   3
18 2013-11-01       19                   3
19 2013-11-01       20                   3
timestamp             240000
grid_id            

In [29]:
multiple_country_rows = grid_hour_counts[
    grid_hour_counts["country_code_count"] > 1
]

print(
    "\nGrid/hour combinations with multiple country codes:",
    len(multiple_country_rows)
)

print(multiple_country_rows.head(20))


Grid/hour combinations with multiple country codes: 236949
    timestamp  grid_id  country_code_count
0  2013-11-01        1                   3
1  2013-11-01        2                   3
2  2013-11-01        3                   3
3  2013-11-01        4                   3
4  2013-11-01        5                   3
5  2013-11-01        6                   3
6  2013-11-01        7                   3
7  2013-11-01        8                   3
8  2013-11-01        9                   3
9  2013-11-01       10                   3
10 2013-11-01       11                   4
11 2013-11-01       12                   4
12 2013-11-01       13                   5
13 2013-11-01       14                   4
14 2013-11-01       15                   3
15 2013-11-01       16                   4
16 2013-11-01       17                   3
17 2013-11-01       18                   3
18 2013-11-01       19                   3
19 2013-11-01       20                   3


In [30]:
if len(multiple_country_rows) > 0:

    example = multiple_country_rows.iloc[0]

    example_rows = df[
        (df["timestamp"] == example["timestamp"]) &
        (df["grid_id"] == example["grid_id"])
    ]

    print("\n===== EXAMPLE =====")
    print(example_rows)


===== EXAMPLE =====
   timestamp  grid_id  country_code  sms_in  sms_out  call_in  call_out  \
0 2013-11-01        1             0  0.3521      NaN      NaN    0.0273   
1 2013-11-01        1            33     NaN      NaN      NaN       NaN   
2 2013-11-01        1            39  1.7322   1.1047   0.5919    0.4020   

   internet_activity        date  hour day_of_week  
0                NaN  2013-11-01     0      Friday  
1             0.0261  2013-11-01     0      Friday  
2            57.7729  2013-11-01     0      Friday  


14. Confirm raw grain

In [31]:

grain_cols = [
    "timestamp",
    "grid_id",
    "country_code"
]

grain_duplicates = df.duplicated(
    subset=grain_cols
).sum()

print("\n===== GRAIN VALIDATION =====")
print("Duplicate rows at expected grain:", grain_duplicates)

if grain_duplicates == 0:
    print(
        "PASS: timestamp + grid_id + country_code "
        "forms a unique raw grain."
    )
else:
    print(
        "FAIL: Duplicate records exist at "
        "timestamp + grid_id + country_code."
    )


===== GRAIN VALIDATION =====
Duplicate rows at expected grain: 0
PASS: timestamp + grid_id + country_code forms a unique raw grain.


15. Create derived activity measures

In [33]:
df["total_sms"] = (
    df["sms_in"].fillna(0)
    + df["sms_out"].fillna(0)
)

In [34]:
df["total_calls"] = (
    df["call_in"].fillna(0)
    + df["call_out"].fillna(0)
)

In [35]:
df["total_activity"] = (
    df["total_sms"]
    + df["total_calls"]
    + df["internet_activity"].fillna(0)
)

In [36]:
print("\n===== DERIVED ACTIVITY MEASURES =====")

print(
    df[
        [
            "sms_in",
            "sms_out",
            "total_sms",
            "call_in",
            "call_out",
            "total_calls",
            "internet_activity",
            "total_activity"
        ]
    ].head()
)


===== DERIVED ACTIVITY MEASURES =====
   sms_in  sms_out  total_sms  call_in  call_out  total_calls  \
0  0.3521      NaN     0.3521      NaN    0.0273       0.0273   
1     NaN      NaN     0.0000      NaN       NaN       0.0000   
2  1.7322   1.1047     2.8369   0.5919    0.4020       0.9939   
3  0.3581      NaN     0.3581      NaN    0.0273       0.0273   
4     NaN      NaN     0.0000      NaN       NaN       0.0000   

   internet_activity  total_activity  
0                NaN          0.3794  
1             0.0261          0.0261  
2            57.7729         61.6037  
3                NaN          0.3854  
4             0.0274          0.0274  


16. Number of unique grids

In [37]:
unique_grids = df["grid_id"].nunique()

print("\n===== UNIQUE GRIDS =====")
print("Number of unique grids:", unique_grids)


===== UNIQUE GRIDS =====
Number of unique grids: 10000


17. Time range

In [38]:
min_timestamp = df["timestamp"].min()
max_timestamp = df["timestamp"].max()

print("\n===== TIME RANGE =====")
print("Start:", min_timestamp)
print("End  :", max_timestamp)


===== TIME RANGE =====
Start: 2013-11-01 00:00:00
End  : 2013-11-01 23:00:00


18. Cadence

In [39]:
print("\n===== CADENCE =====")

if cadence_valid:
    print("Hourly")
else:
    print("Irregular / invalid")


===== CADENCE =====
Hourly


19. Country-code categories

In [40]:
country_codes = sorted(
    df["country_code"]
    .dropna()
    .unique()
)

print("\n===== COUNTRY CODE CATEGORIES =====")
print("Number of country-code categories:", len(country_codes))
print("Country codes:", country_codes)


===== COUNTRY CODE CATEGORIES =====
Number of country-code categories: 246
Country codes: [np.int64(0), np.int64(1), np.int64(7), np.int64(20), np.int64(27), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(36), np.int64(39), np.int64(40), np.int64(41), np.int64(43), np.int64(44), np.int64(45), np.int64(46), np.int64(47), np.int64(48), np.int64(49), np.int64(51), np.int64(52), np.int64(53), np.int64(54), np.int64(55), np.int64(56), np.int64(57), np.int64(58), np.int64(60), np.int64(61), np.int64(62), np.int64(63), np.int64(64), np.int64(65), np.int64(66), np.int64(81), np.int64(82), np.int64(84), np.int64(86), np.int64(90), np.int64(91), np.int64(92), np.int64(93), np.int64(94), np.int64(95), np.int64(98), np.int64(211), np.int64(212), np.int64(213), np.int64(216), np.int64(218), np.int64(220), np.int64(221), np.int64(222), np.int64(223), np.int64(224), np.int64(225), np.int64(226), np.int64(227), np.int64(228), np.int64(229), np.int64(230), np.int64(231)

In [41]:
country_code_counts = (
    df["country_code"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\n===== COUNTRY CODE FREQUENCY =====")
print(country_code_counts)


===== COUNTRY CODE FREQUENCY =====
country_code
0        231307
1         42777
7         27509
20        78182
27         1260
          ...  
18768        13
68572        28
88235         9
88239      4259
97259        51
Name: count, Length: 246, dtype: Int64


20. Busiest hourly window

In [42]:
hourly_activity = (
    df.groupby("hour")["total_activity"]
    .sum()
    .sort_values(ascending=False)
)

print("\n===== HOURLY ACTIVITY =====")
print(hourly_activity)


===== HOURLY ACTIVITY =====
hour
11    5.185704e+06
18    5.135660e+06
17    5.108601e+06
12    5.038444e+06
10    4.926999e+06
16    4.916011e+06
19    4.912902e+06
15    4.808330e+06
14    4.774330e+06
20    4.700756e+06
13    4.651159e+06
21    4.506839e+06
9     4.211152e+06
22    4.151605e+06
0     3.946939e+06
23    3.760764e+06
8     3.322538e+06
1     3.299749e+06
2     2.821881e+06
7     2.525391e+06
3     2.488233e+06
4     2.253658e+06
6     2.170194e+06
5     2.109789e+06
Name: total_activity, dtype: float64


In [43]:
busiest_hour = hourly_activity.idxmax()
busiest_hour_value = hourly_activity.max()

print("\nBusiest hour:", busiest_hour)
print("Total activity:", busiest_hour_value)


Busiest hour: 11
Total activity: 5185703.659


In [44]:
df.head()

,timestamp,grid_id,country_code,sms_in,sms_out,call_in,call_out,internet_activity,date,hour,day_of_week,total_sms,total_calls,total_activity
0,2013-11-01,1,0,0.3521,NaN,NaN,0.0273,NaN,2013-11-01,0,Friday,0.3521,0.0273,0.3794
1,2013-11-01,1,33,NaN,NaN,NaN,NaN,0.0261,2013-11-01,0,Friday,0.0000,0.0000,0.0261
2,2013-11-01,1,39,1.7322,1.1047,0.5919,0.4020,57.7729,2013-11-01,0,Friday,2.8369,0.9939,61.6037
3,2013-11-01,2,0,0.3581,NaN,NaN,0.0273,NaN,2013-11-01,0,Friday,0.3581,0.0273,0.3854
4,2013-11-01,2,33,NaN,NaN,NaN,NaN,0.0274,2013-11-01,0,Friday,0.0000,0.0000,0.0274


21. Busiest grid

In [45]:
grid_activity = (
    df.groupby("grid_id")["total_activity"]
    .sum()
    .sort_values(ascending=False)
)

busiest_grid = grid_activity.idxmax()
busiest_grid_value = grid_activity.max()

print("\n===== BUSIEST GRID =====")
print("Grid:", busiest_grid)
print("Total activity:", busiest_grid_value)


===== BUSIEST GRID =====
Grid: 5161
Total activity: 274800.2956


22. Null counts per column

In [46]:

null_counts = (
    df.isna()
    .sum()
    .sort_values(ascending=False)
)

print("\n===== NULL COUNTS PER COLUMN =====")
print(null_counts)


===== NULL COUNTS PER COLUMN =====
sms_out              1422446
call_in              1407781
internet_activity    1087074
sms_in               1086153
call_out             1037413
timestamp                  0
country_code               0
grid_id                    0
date                       0
hour                       0
day_of_week                0
total_sms                  0
total_calls                0
total_activity             0
dtype: int64


In [47]:
null_profile = pd.DataFrame({
    "column": df.columns,
    "null_count": [
        df[col].isna().sum()
        for col in df.columns
    ],
    "null_percentage": [
        round(df[col].isna().mean() * 100, 2)
        for col in df.columns
    ]
})

print("\n===== NULL PROFILE =====")
print(null_profile)


===== NULL PROFILE =====
               column  null_count  null_percentage
0           timestamp           0             0.00
1             grid_id           0             0.00
2        country_code           0             0.00
3              sms_in     1086153            57.41
4             sms_out     1422446            75.18
5             call_in     1407781            74.41
6            call_out     1037413            54.83
7   internet_activity     1087074            57.46
8                date           0             0.00
9                hour           0             0.00
10        day_of_week           0             0.00
11          total_sms           0             0.00
12        total_calls           0             0.00
13     total_activity           0             0.00


23. Create one complete profiling report

In [48]:

profiling_facts = {
    "row_count": len(df),
    "column_count": len(df.columns),
    "unique_grids": df["grid_id"].nunique(),
    "time_start": str(df["timestamp"].min()),
    "time_end": str(df["timestamp"].max()),
    "distinct_timestamps": len(unique_timestamps),
    "hourly_cadence": cadence_valid,
    "country_code_categories": country_codes,
    "busiest_hour": int(busiest_hour),
    "busiest_hour_total_activity": float(busiest_hour_value),
    "busiest_grid": int(busiest_grid),
    "busiest_grid_total_activity": float(busiest_grid_value),
    "exact_duplicate_rows": int(duplicate_count),
    "duplicate_rows_at_expected_grain": int(grain_duplicates)
}

print("\n===== PROFILING FACTS =====")

for key, value in profiling_facts.items():
    print(f"{key}: {value}")


===== PROFILING FACTS =====
row_count: 1891928
column_count: 14
unique_grids: 10000
time_start: 2013-11-01 00:00:00
time_end: 2013-11-01 23:00:00
distinct_timestamps: 24
hourly_cadence: True
country_code_categories: [np.int64(0), np.int64(1), np.int64(7), np.int64(20), np.int64(27), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(36), np.int64(39), np.int64(40), np.int64(41), np.int64(43), np.int64(44), np.int64(45), np.int64(46), np.int64(47), np.int64(48), np.int64(49), np.int64(51), np.int64(52), np.int64(53), np.int64(54), np.int64(55), np.int64(56), np.int64(57), np.int64(58), np.int64(60), np.int64(61), np.int64(62), np.int64(63), np.int64(64), np.int64(65), np.int64(66), np.int64(81), np.int64(82), np.int64(84), np.int64(86), np.int64(90), np.int64(91), np.int64(92), np.int64(93), np.int64(94), np.int64(95), np.int64(98), np.int64(211), np.int64(212), np.int64(213), np.int64(216), np.int64(218), np.int64(220), np.int64(221), np.int64(222), np.int6

24. Final five-bullet summary

In [49]:
# ------------------------------------------------------------
# Five-bullet profiling summary
# ------------------------------------------------------------

summary = [
    (
        f"• The supplied file contains {len(df):,} rows across "
        f"{df['grid_id'].nunique():,} unique grids, covering "
        f"{df['timestamp'].min()} to {df['timestamp'].max()}."
    ),

    (
        f"• The file contains {len(unique_timestamps)} distinct hourly "
        f"timestamps, and the consecutive timestamp intervals are "
        f"{'exactly one hour apart' if cadence_valid else 'not consistently one hour apart'}."
    ),

    (
        f"• The raw activity grain was validated as "
        f"timestamp + grid_id + country_code; "
        f"{len(multiple_country_rows):,} grid/hour combinations contain "
        f"more than one country-code row."
    ),

    (
        f"• The busiest hourly window is "
        f"{busiest_hour:02d}:00–{busiest_hour:02d}:59, while the busiest "
        f"grid is {busiest_grid}, based on total_activity."
    ),

    (
        f"• Data-quality profiling identified {duplicate_count:,} exact "
        f"duplicate rows and {grain_duplicates:,} duplicate records at "
        f"the expected grain; null counts were profiled separately "
        f"without modifying the raw source file."
    )
]

print("\n===== NETWORK ANALYTICS TEAM — DATA PROFILING SUMMARY =====")

for bullet in summary:
    print(bullet)


===== NETWORK ANALYTICS TEAM — DATA PROFILING SUMMARY =====
• The supplied file contains 1,891,928 rows across 10,000 unique grids, covering 2013-11-01 00:00:00 to 2013-11-01 23:00:00.
• The file contains 24 distinct hourly timestamps, and the consecutive timestamp intervals are exactly one hour apart.
• The raw activity grain was validated as timestamp + grid_id + country_code; 236,949 grid/hour combinations contain more than one country-code row.
• The busiest hourly window is 11:00–11:59, while the busiest grid is 5161, based on total_activity.
• Data-quality profiling identified 0 exact duplicate rows and 0 duplicate records at the expected grain; null counts were profiled separately without modifying the raw source file.
